In [ ]:
import numpy as np

class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.lr = lr

        # Initialize weights and biases
        self.Wh = np.random.randn(hidden_size, hidden_size) * 0.1
        self.Wx = np.random.randn(hidden_size, input_size) * 0.1
        self.Wy = np.random.randn(output_size, hidden_size) * 0.1

        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, X):
        h = np.zeros((self.Wh.shape[0], 1))
        self.hidden_states = [h]

        for x in X:
            x = x.reshape(-1, 1)

            h = np.tanh(self.Wx @ x + self.Wh @ h + self.bh)
            self.hidden_states.append(h)

        self.output = self.Wy @ h + self.by

        return self.output

    def backward(self, X, y):
        # Calculate output error
        error = self.output - y.reshape(-1, 1)

        dWy = error @ self.hidden_states[-1].T
        dby = error

        dh = self.Wy.T @ error

        dWx = np.zeros_like(self.Wx)
        dWh = np.zeros_like(self.Wh)
        dbh = np.zeros_like(self.bh)

        # Backpropagation through time
        for t in reversed(range(len(X))):
            h = self.hidden_states[t + 1]
            h_prev = self.hidden_states[t]
            x = X[t].reshape(-1, 1)

            dz = dh * (1 - h ** 2)

            dWx += dz @ x.T
            dWh += dz @ h_prev.T
            dbh += dz

            dh = self.Wh.T @ dz

        # Update weights and biases
        self.Wy -= self.lr * dWy
        self.by -= self.lr * dby

        self.Wx -= self.lr * dWx
        self.Wh -= self.lr * dWh
        self.bh -= self.lr * dbh

    def train(self, X, y, epochs=100):
        for epoch in range(epochs):
            loss = 0

            for xi, yi in zip(X, y):
                prediction = self.forward(xi)

                target = yi.reshape(-1, 1)
                loss += np.mean((prediction - target) ** 2)

                self.backward(xi, yi)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, Loss: {loss / len(X):.4f}")

    def predict(self, X):
        return self.forward(X)


# Sample data
X = np.array([
    [[1], [2], [3]],
    [[2], [3], [4]],
    [[3], [4], [5]],
    [[4], [5], [6]]
], dtype=float)

y = np.array([
    [4],
    [5],
    [6],
    [7]
], dtype=float)

# Create and train the RNN
rnn = SimpleRNN(input_size=1, hidden_size=4, output_size=1, lr=0.001)

rnn.train(X, y, epochs=100)

# Make a prediction
test = np.array([[5], [6], [7]], dtype=float)

prediction = rnn.predict(test)

print("\nPredicted next value:", prediction.item())

Epoch 10, Loss: 26.2925
Epoch 20, Loss: 20.5719
Epoch 30, Loss: 15.1539
Epoch 40, Loss: 10.8049
Epoch 50, Loss: 7.6587
Epoch 60, Loss: 5.4867
Epoch 70, Loss: 4.0123
Epoch 80, Loss: 3.0176
Epoch 90, Loss: 2.3480
Epoch 100, Loss: 1.8969

Predicted next value: 4.708622973707174
